# Variance swap

Static replication of a variance swap with a strip of OTM options of a single maturity $T$, using 10 strikes from the 10d put to the 10d call:

$$\sigma^2_{\text{realized}} \approx \frac{2}{T}\int_0^{\infty} \frac{O(K)}{K^2}\,dK \;\;\Rightarrow\;\; w_i = \frac{2}{T}\,\frac{\Delta K_i}{K_i^2}$$

with $O(K)$ the OTM option (put for $K<F$, call for $K \ge F$) and $\Delta K_i = (K_{i+1}-K_{i-1})/2$ (one-sided at the ends).

With only 10 strikes cut at the 10d wings, these weights miss the gamma of the wings, so the function also fits weights by least squares to make the portfolio gamma cash as flat as possible across the strip and over every evaluation time, either non-negative (`method="nnls"`) or unconstrained (`method="lstsq"`). The proof uses the weights of the chosen `method`.

**Proof**: the portfolio replicates the log contract, whose cash gamma is constant: $\Gamma \cdot S^2 = 2/T$ for every spot and every time. The bubble plot (x = time to maturity, y = spot level, size and colour = value) shows the portfolio gamma cash $\Gamma \cdot S^2 / 100$ (target $2/(100\,T)$) with spot on each strike for each time to maturity in `t_bd` (business days).

## Import

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import nnls
import plotly.graph_objects as go

## Custom functions

In [2]:
def bs_gamma(S, K, t, sigma, r=0.0, q=0.0):
    """Black-Scholes gamma (broadcast over S, K, sigma)."""
    sqrt_t = np.sqrt(t)
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * t) / (sigma * sqrt_t)
    return np.exp(-q * t) * norm.pdf(d1) / (S * sigma * sqrt_t)


def bs_delta(S, K, t, sigma, cp, r=0.0, q=0.0):
    """Black-Scholes spot delta, cp = +1 call / -1 put."""
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * t) / (sigma * np.sqrt(t))
    return cp * np.exp(-q * t) * norm.cdf(cp * d1)


def variance_swap_weights(options, spot, T, r=0.0, q=0.0, t_bd=(20, 19, 18, 17, 16, 15), n_strikes=10, delta_cut=0.10, bd_per_year=252,
                          method="nnls"):
    """
    options   : DataFrame of a single maturity with columns 'strike', 'type' ('C'/'P'), 'iv'
    spot      : underlying spot
    T         : maturity in years
    t_bd      : times to maturity (in business days) at which the portfolio is evaluated in the proof
    n_strikes : number of strikes kept in the strip
    delta_cut : wings of the strip (10d put to 10d call by default)
    bd_per_year : business days per year to convert t_bd to years
    method    : weights used in the proof: "theory", "nnls" (non-negative least squares)
                or "lstsq" (ordinary least squares, negative weights allowed)

    Returns (weights, fig):
      weights : DataFrame of the OTM options with the theoretical weight 2*dK/(T*K^2) and the weights fitted on
                flat gamma cash over the strip and all t_bd (nnls and lstsq), per 1 unit of variance notional
      fig     : plotly bubble plot of the portfolio gamma cash, x = t_bd, y = spot level (= strikes)
    """
    df = options.copy()
    df["type"] = df["type"].str.upper().str[0]
    F = spot * np.exp((r - q) * T)

    # keep OTM options: puts below the forward, calls at/above
    otm = ((df["type"] == "P") & (df["strike"] < F)) | ((df["type"] == "C") & (df["strike"] >= F))
    df = df[otm].sort_values("strike").reset_index(drop=True)

    # keep strikes between the 10d put and the 10d call
    cp = np.where(df["type"] == "C", 1.0, -1.0)
    df["delta"] = bs_delta(spot, df["strike"], T, df["iv"], cp, r, q)
    df = df[df["delta"].abs() >= delta_cut].reset_index(drop=True)

    # pick n_strikes evenly spaced across that range (wings included)
    idx = np.unique(np.round(np.linspace(0, len(df) - 1, n_strikes)).astype(int))
    df = df.iloc[idx].reset_index(drop=True)

    K = df["strike"].to_numpy(float)
    iv = df["iv"].to_numpy(float)
    dK = np.gradient(K)
    df["weight_theory"] = 2.0 * dK / (T * K**2)

    # fitted weights: with a truncated strip the theoretical weights lose the wings' gamma,
    # so solve min ||sum_i w_i * gamma_cash_i(S, t) - 2/(100T)|| over a fine spot grid
    # across the strip and every evaluation time, with w >= 0 (nnls) or unconstrained (lstsq)
    target = 2 / (100 * T)
    S_fit = np.linspace(K[0], K[-1], 101)[:, None]
    A = np.vstack([bs_gamma(S_fit, K[None, :], bd / bd_per_year, iv[None, :], r, q) * S_fit**2 / 100 for bd in t_bd])
    b = np.full(len(A), target)
    df["weight_nnls"], _ = nnls(A, b)
    df["weight_lstsq"] = np.linalg.lstsq(A, b, rcond=None)[0]
    weights = df[["strike", "type", "iv", "delta", "weight_theory", "weight_nnls", "weight_lstsq"]]
    w = df[f"weight_{method}"].to_numpy()

    # proof: portfolio gamma cash with spot on each strike, for each bumped time to maturity
    S = K[:, None]
    gamma_cash = np.column_stack([
        (bs_gamma(S, K[None, :], bd / bd_per_year, iv[None, :], r, q) * S**2 / 100) @ w for bd in t_bd
    ])  # rows = spot levels (strikes), columns = t_bd
    x = np.tile(np.asarray(t_bd), len(K))
    y = np.repeat(K, len(t_bd))
    z = gamma_cash.ravel()

    fig = go.Figure(go.Scatter(
        x=x, y=y, mode="markers",
        marker=dict(size=np.abs(z), sizemode="area", sizeref=2 * np.abs(z).max() / 40**2, sizemin=2,
                    color=z, colorscale="RdBu", cmid=target, showscale=True,
                    colorbar=dict(title="Gamma cash")),
        customdata=np.column_stack([z, (z / target - 1) * 100]),
        hovertemplate="t=%{x}bd<br>spot=%{y}<br>gamma cash=%{customdata[0]:.4f}<br>vs target=%{customdata[1]:+.2f}%<extra></extra>",
    ))
    fig.update_layout(title=f"Variance swap portfolio gamma cash ({method} weights, target 2/(100T) = {target:.4f})",
                      xaxis=dict(title="Time to maturity (business days)", autorange="reversed", tickvals=list(t_bd)),
                      yaxis=dict(title="Spot level"))
    return weights, fig

## Example

In [3]:
spot, T, r, q = 100.0, 21 / 252, 0.03, 0.0   # 1-month option (21 business days)
F = spot * np.exp((r - q) * T)

strikes = np.arange(70, 131, 1.0)
iv = 0.20 - 0.10 * np.log(strikes / F) + 0.30 * np.log(strikes / F) ** 2   # synthetic smile
options = pd.concat([
    pd.DataFrame({"strike": strikes, "type": "C", "iv": iv}),
    pd.DataFrame({"strike": strikes, "type": "P", "iv": iv}),
], ignore_index=True)

weights, fig_nnls = variance_swap_weights(options, spot, T, r, q, t_bd=[20, 19, 18, 17, 16, 15], method="nnls")
_, fig_lstsq = variance_swap_weights(options, spot, T, r, q, t_bd=[20, 19, 18, 17, 16, 15], method="lstsq")
weights

,strike,type,iv,delta,weight_theory,weight_nnls,weight_lstsq
0,93.0,P,0.209198,-0.101458,0.005550,0.028209,1.425301
1,95.0,P,0.206247,-0.175339,0.003989,0.000000,-11.039003
2,96.0,P,0.204895,-0.223028,0.003906,0.000000,16.207974
3,98.0,P,0.202425,-0.338072,0.003748,0.000000,-8.161083
4,99.0,P,0.201302,-0.403218,0.003673,0.007924,-7.940242
5,101.0,C,0.199272,0.459873,0.003529,0.002578,41.993750
6,102.0,C,0.198360,0.392220,0.003460,0.000000,-47.235389
7,104.0,C,0.196732,0.268225,0.003328,0.000000,29.737277
8,105.0,C,0.196014,0.214828,0.003265,0.000000,-16.528553
9,107.0,C,0.194758,0.129060,0.004193,0.023075,1.627342


In [4]:
fig_nnls.show()

In [5]:
fig_lstsq.show()